# 05 — Cost Analysis

Compare training time, tuning cost, and the performance-vs-compute trade-off.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14, 'savefig.dpi': 300})

os.makedirs('../results/figures', exist_ok=True)

%matplotlib inline

test_df = pd.read_csv('../results/aggregated/test_results.csv')
fold_df = pd.read_csv('../results/aggregated/fold_results.csv')

In [ ]:
# Training time comparison
if 'train_time_s' in fold_df.columns:
    time_summary = fold_df.groupby('model')['train_time_s'].agg(['mean', 'std', 'sum'])
    time_summary = time_summary.sort_values('mean')

    fig, ax = plt.subplots(figsize=(10, 6))
    time_summary['mean'].plot.barh(ax=ax, xerr=time_summary['std'],
                                    color='steelblue', capsize=3)
    ax.set_xlabel('Mean Training Time per Fold (seconds)')
    ax.set_title('Average Training Time per Model')
    plt.tight_layout()
    plt.savefig('../results/figures/training_time.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("\nTraining time summary (seconds):")
    display(time_summary)

In [ ]:
# Tuning cost (total time including Optuna trials)
if 'tuning_time_s' in test_df.columns:
    tuning_summary = test_df.groupby('model')['tuning_time_s'].agg(['mean', 'sum'])
    tuning_summary = tuning_summary.sort_values('mean')

    fig, ax = plt.subplots(figsize=(10, 6))
    tuning_summary['mean'].plot.barh(ax=ax, color='coral')
    ax.set_xlabel('Mean Tuning Time per Dataset (seconds)')
    ax.set_title('Hyperparameter Tuning Cost')
    plt.tight_layout()
    plt.savefig('../results/figures/tuning_cost.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Performance vs training time scatter (Pareto front)
model_families = {
    'xgboost': 'GBDT', 'lightgbm': 'GBDT', 'catboost': 'GBDT',
    'ft_transformer': 'DL', 'tabnet': 'DL', 'saint': 'DL', 'tabm': 'DL',
    'mlp': 'DL', 'realmlp': 'DL',
    'tabpfn': 'Foundation',
}

merged = test_df.copy()
merged['family'] = merged['model'].map(model_families)

for task_type, metric in [('binary', 'roc_auc'), ('multiclass', 'accuracy'), ('regression', 'r2')]:
    subset = merged[(merged['task_type'] == task_type) & (metric in merged.columns)]
    if subset.empty or metric not in subset.columns:
        continue

    fig, ax = plt.subplots(figsize=(10, 6))
    for model, group in subset.groupby('model'):
        ax.scatter(group['train_time_s'], group[metric],
                  label=model, s=80, alpha=0.8)
    ax.set_xscale('log')
    ax.set_xlabel('Training Time (seconds, log scale)')
    ax.set_ylabel(metric)
    ax.set_title(f'{task_type.title()}: Performance vs Training Cost')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'../results/figures/pareto_{task_type}.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Total compute budget table
if 'train_time_s' in test_df.columns and 'tuning_time_s' in test_df.columns:
    total_time = test_df.groupby('model').agg({
        'train_time_s': 'sum',
        'tuning_time_s': 'sum',
    }).rename(columns={
        'train_time_s': 'total_train_s',
        'tuning_time_s': 'total_tune_s',
    })
    total_time['total_s'] = total_time['total_train_s'] + total_time['total_tune_s']
    total_time['total_hours'] = total_time['total_s'] / 3600
    total_time = total_time.sort_values('total_s')

    print("\nTotal Compute Budget:")
    display(total_time)
else:
    print("Timing columns (train_time_s / tuning_time_s) not found in test_df.")